# Hyrcanian forest diversity: Earth Engine feature extraction

Public-facing companion to the forest-diversity manuscript and `Diversity_Embedding_DNN_spatial_CV.ipynb`. This adapts the supplied Earth Engine extraction code. Inventory records are restricted by the data provider: **do not commit field points, coordinates, attributes, export CSVs, or task logs.** Outputs remain in your own Google Drive and must be downloaded into a private working directory.

**Workflow:** authorize Earth Engine → privately load plot IDs and coordinates plus study-area geometry → export 2023 Sentinel-1, Sentinel-2, RaoQ, and AlphaEarth plot features → separately obtain the licensed/requested TESSERA tiles → audit and merge the exported tables locally. The original code uses 10 m point sampling, whereas the manuscript says 17.84 m circular plot means. This version computes circular plot means for every product; it cannot guarantee numerical identity to previously reported results. See the checks at the end.

## 1. Settings and restricted inputs

Install `earthengine-api`, `geopandas`, `pandas`, `numpy`, `shapely`, and `jupyter` in your environment. Authenticate with `earthengine authenticate` before running, or authenticate interactively in the setup cell. Provide your own Earth Engine Cloud project and private paths using environment variables.

In [ ]:
import os
from pathlib import Path
import ee
import geopandas as gpd
import pandas as pd
import numpy as np

EE_PROJECT = os.environ.get('EE_PROJECT')
PLOTS_CSV = Path(os.environ.get('DIVERSITY_PLOTS_CSV', 'private/plots.csv'))
AOI_PATH = Path(os.environ.get('DIVERSITY_AOI_FILE', 'private/study_area.gpkg'))
EXPORT_FOLDER = os.environ.get('DIVERSITY_EXPORT_FOLDER', 'EarthEngineExports')
PLOT_RADIUS_M = 17.84
start_date, end_date = '2023-01-01', '2024-01-01'
BIN_DAYS = 16
if not EE_PROJECT:
    raise ValueError('Set EE_PROJECT to your own Earth Engine-enabled Cloud project.')
try:
    ee.Initialize(project=EE_PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)
for path in (PLOTS_CSV, AOI_PATH):
    if not path.exists():
        raise FileNotFoundError(f'Provide the private input at {path}')

## 2. Load plots and the study area

Only `PLOTID` is transmitted as a property with geometries for the Earth Engine computation. The four management-zone labels required by the modeling notebook stay in the private plot table; add them during the later local join. Coordinate reference system of the input plot CSV must be WGS84 longitude/latitude.

In [ ]:
plots = pd.read_csv(PLOTS_CSV)
required = ['PLOTID','Longitude','Latitude','management_zone']
missing = [k for k in required if k not in plots]
if missing:
    raise ValueError(f'Missing private plot columns: {missing}')
if plots['PLOTID'].isna().any() or plots['PLOTID'].duplicated().any():
    raise ValueError('PLOTID must identify exactly one plot.')
for col in ['Longitude','Latitude']:
    plots[col] = pd.to_numeric(plots[col],errors='raise')
if not plots.Longitude.between(-180,180).all() or not plots.Latitude.between(-90,90).all():
    raise ValueError('Longitude/Latitude must be WGS84 degrees.')
if plots.management_zone.isna().any() or plots.management_zone.nunique() != 4:
    raise ValueError('The manuscript requires four genuine management-zone labels.')
aoi = gpd.read_file(AOI_PATH)
if aoi.crs is None:
    raise ValueError('Study-area geometry has no CRS: assign its true CRS upstream.')
aoi = aoi.to_crs('EPSG:4326')
aoi = aoi[aoi.geometry.notna() & ~aoi.geometry.is_empty]
if aoi.empty:
    raise ValueError('Empty study-area geometry')
from shapely.geometry import mapping
edaratkol_fc = ee.FeatureCollection([ee.Feature(ee.Geometry(mapping(geom))) for geom in aoi.geometry])
table_field_data = ee.FeatureCollection([
    ee.Feature(ee.Geometry.Point([float(r.Longitude),float(r.Latitude)]),
               {'PLOTID': str(r.PLOTID)}) for r in plots.itertuples(index=False)])
plot_buffers = table_field_data.map(lambda f: f.buffer(PLOT_RADIUS_M))
print('Plot count:',len(plots),'management-zone counts:',plots.management_zone.value_counts().to_dict())

## 3. Export helper: circular plot mean for each 16-day image

Each call starts one Drive export per nonempty bin. Check Earth Engine task status and the number of completed files before merging. Empty or masked pixels can yield missing output rows; investigate rather than assigning zeros.

In [ ]:
def export_bin_means(collection, prefix, scale=10, tile_scale=16):
    image_list = collection.toList(collection.size())
    count = int(collection.size().getInfo())
    tasks = []
    for i in range(count):
        img = ee.Image(image_list.get(i))
        # 17.84 m circular mean for each individual plot; retains PLOTID.
        rows = img.reduceRegions(collection=plot_buffers,
                                 reducer=ee.Reducer.mean(),scale=scale,tileScale=tile_scale)
        midpoint = ee.Date(img.get('system:time_start'))
        begin = midpoint.advance(-BIN_DAYS/2, 'day')
        finish = begin.advance(BIN_DAYS-1, 'day')
        rows = rows.map(lambda f: f.set({'bin':begin.format('YYYY-MM-dd').cat('_to_').cat(finish.format('YYYY-MM-dd')),
                                        'bin_start':begin.format('YYYY-MM-dd'),
                                        'bin_end':finish.format('YYYY-MM-dd'),
                                        'n_imgs':img.get('n_imgs')}))
        description = f'{prefix}_bin_{i:02d}'
        task = ee.batch.Export.table.toDrive(collection=rows,description=description,
                    folder=EXPORT_FOLDER,fileNamePrefix=description,fileFormat='CSV',
                    selectors=['PLOTID','bin','bin_start','bin_end','n_imgs']+
                              img.bandNames().getInfo())
        task.start()
        tasks.append(task)
    print(f'Started {len(tasks)} tasks with prefix {prefix}. Wait for completion.')
    return tasks

## Sentinel-1

The preprocessing logic below is adapted from the supplied notebook. Review its methods against the final manuscript before claiming exact numerical replication.

In [ ]:
import ee, math

# =====================================================
# PARAMETERS
# =====================================================

SCALE      = 10
TILESCALE  = 16          # must be 1..16
BUFFER_M   = 60          # around points

# --- S1 ND TEXTURES (memory-safe defaults) ---
TEXTURE_SIZE  = 3        # 3x3 window (lighter than 5x5)
TEXTURE_SCALE = 20       # compute textures at 20 m to reduce memory

aoi_geom   = edaratkol_fc.geometry()
points_fc  = table_field_data

pols = ['VV','VH']
eps  = 1e-6

# =====================================================
# IMPORTANT: use a SIMPLE region around points
# (bounds() makes it a simple rectangle => far less memory)
# =====================================================
pts_bbox = points_fc.geometry().buffer(BUFFER_M).bounds()

# =====================================================
# BASE S1 COLLECTION (limit early)
# =====================================================
s1 = (ee.ImageCollection('COPERNICUS/S1_GRD')
      .filterBounds(pts_bbox)                     # <<< limit early
      .filterDate(start_date, end_date)
      .filter(ee.Filter.eq('instrumentMode', 'IW'))
      .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
      .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH')))

# =====================================================
# TERRAIN (precompute once)
# =====================================================
dem  = ee.Image('USGS/SRTMGL1_003').clip(pts_bbox)
terr = ee.Terrain.products(dem)
slope  = terr.select('slope')
aspect = terr.select('aspect')

pi  = ee.Number(math.pi)
rad = pi.divide(ee.Number(180.0))

# =====================================================
# FUNCTIONS
# =====================================================
def to_linear(img):
    # VV/VH in linear + keep angle
    return (ee.Image(10).pow(img.select(pols).divide(10))
            .addBands(img.select('angle'))
            .copyProperties(img, img.propertyNames()))

def speckle_mean(img):
    # compute only in small bbox
    img = img.clip(pts_bbox)
    return (img.select(pols).focal_mean(radius=3, kernelType='circle', iterations=1)
            .addBands(img.select('angle'))
            .copyProperties(img, img.propertyNames()))

def slope_correction_gamma0(img, pass_name):
    # use precomputed slope/aspect
    theta_i = img.select('angle').multiply(rad)
    slope_r = slope.multiply(rad)
    aspect_r = aspect.multiply(rad)

    phi_i = ee.Image.constant(0.0) if pass_name == 'ASCENDING' else ee.Image.constant(pi)
    phi_r = phi_i.subtract(aspect_r)

    local_lia = (theta_i.cos().multiply(slope_r.cos())
                 .add(theta_i.sin().multiply(slope_r.sin()).multiply(phi_r.cos()))
                 ).acos()

    sigma0_lin = img.select(pols)
    gamma0_lin = sigma0_lin.divide(local_lia.cos())
    gamma0_dB  = gamma0_lin.log10().multiply(10)

    out = ee.Image.cat(
        gamma0_lin.rename(['VV_gamma0','VH_gamma0']),
        gamma0_dB.rename(['VV_gamma0_dB','VH_gamma0_dB'])
    )
    return out.copyProperties(img, img.propertyNames())

def add_s1_indices(img):
    vv = img.select('VV_gamma0')
    vh = img.select('VH_gamma0')
    nd = vv.subtract(vh).divide(vv.add(vh).add(eps)).rename('S1_ND')
    ratio = vv.divide(vh.add(eps)).rename('S1_VV_over_VH')
    return img.addBands([nd, ratio])

def add_nd_textures(img):
    """
    Adds GLCM textures computed from S1_ND (on the binned composite).
    Outputs: ND_CON, ND_DIS, ND_ENT, ND_HOM

    Memory-safe approach:
      - clip to pts_bbox
      - quantize ND to Uint8 [0..255]
      - compute textures at coarser scale (TEXTURE_SCALE)
      - keep only needed texture bands
    """
    img = img.clip(pts_bbox)

    nd = img.select('S1_ND')

    # Quantize ND (~[-1,1]) into uint8 (0..255)
    nd8 = (nd.clamp(-1, 1)
             .unitScale(-1, 1)
             .multiply(255)
             .toUint8()
             .rename('ND_u8'))

    # Coarser reprojection for texture computation
    nd8_tex = nd8.reproject(nd8.projection().atScale(TEXTURE_SCALE))

    tex = nd8_tex.glcmTexture(size=TEXTURE_SIZE)

    out = (tex.select(['ND_u8_contrast', 'ND_u8_diss', 'ND_u8_ent', 'ND_u8_idm'])
              .rename(['ND_CON', 'ND_DIS', 'ND_ENT', 'ND_HOM']))

    return img.addBands(out)

def build_s1_per_acq():
    def per_pass(pass_name):
        return (s1.filter(ee.Filter.eq('orbitProperties_pass', pass_name))
                .map(to_linear)
                .map(speckle_mean)  # already clipped to bbox
                .map(lambda im: slope_correction_gamma0(im, pass_name))
                .map(add_s1_indices)
                .map(lambda im: im.set('orbit', pass_name))
                .select(['VV_gamma0','VH_gamma0','VV_gamma0_dB','VH_gamma0_dB','S1_ND','S1_VV_over_VH'])
               )
    return per_pass('ASCENDING').merge(per_pass('DESCENDING')).sort('system:time_start')

s1_ts = build_s1_per_acq()

# =====================================================
# 16-DAY BINNING (median per bin) + ND TEXTURES
# =====================================================
start = ee.Date(start_date)
end   = ee.Date(end_date)
n_bins = end.difference(start, 'day').divide(BIN_DAYS).ceil()
bins = ee.List.sequence(0, n_bins.subtract(1))

def bin_to_image(i):
    i = ee.Number(i)
    b_start = start.advance(i.multiply(BIN_DAYS), 'day')
    b_end   = b_start.advance(BIN_DAYS, 'day')
    ic_bin  = s1_ts.filterDate(b_start, b_end)

    # 16-day median composite, clipped early
    comp = ic_bin.median().clip(pts_bbox)

    # Add ND textures on the binned composite (much less memory than per acquisition)
    comp = add_nd_textures(comp)

    bin_label = b_start.format('YYYY-MM-dd').cat('_to_').cat(b_end.advance(-1, 'day').format('YYYY-MM-dd'))
    bin_mid   = b_start.advance(BIN_DAYS/2, 'day')

    return (comp
            .set('bin', bin_label)
            .set('bin_start', b_start.format('YYYY-MM-dd'))
            .set('bin_end', b_end.advance(-1, 'day').format('YYYY-MM-dd'))
            .set('n_imgs', ic_bin.size())
            .set('system:time_start', bin_mid.millis()))

s1_16d = (ee.ImageCollection(bins.map(bin_to_image))
          .filter(ee.Filter.gt('n_imgs', 0))
          .sort('system:time_start'))

# =====================================================


In [ ]:
tasks_14 = export_bin_means(s1_16d, 'S1_2023', scale=10)

## Sentinel-2 spectral, EVI and textures

The preprocessing logic below is adapted from the supplied notebook. Review its methods against the final manuscript before claiming exact numerical replication.

In [ ]:
import ee
# Initialize Earth Engine

# =====================================================
# REQUIRED INPUTS
# =====================================================

BUFFER_M   = 60
# Make sure table_field_data is defined in your notebook prior to running this
points_fc  = table_field_data

# Simple bbox around points (used ONLY to filter the collection, not to clip!)
pts_bbox   = points_fc.geometry().buffer(BUFFER_M).bounds()

# =====================================================
# PARAMETERS
# =====================================================
S2_SCALE_SAMPLE = 10
S2_TILESCALE    = 16

EVI_TEX_SIZE    = 3
EVI_TEX_SCALE   = 10
eps = 1e-6

# EE numbers for date math
BIN_DAYS_EE       = ee.Number(BIN_DAYS)
HALF_BIN_DAYS_EE  = BIN_DAYS_EE.divide(2)
NEG_ONE_DAY_EE    = ee.Number(-1)

# =====================================================
# S2 CLOUD MASK (S2 SR HARMONIZED) + SCALE TO 0..1
# =====================================================
def mask_s2_sr_harmonized(img):
    scl = img.select('SCL')

    # mask cloud/shadow/cirrus/snow
    scl_ok = (scl.neq(3)
              .And(scl.neq(8))
              .And(scl.neq(9))
              .And(scl.neq(10))
              .And(scl.neq(11)))

    qa = img.select('QA60')
    cloud_bit  = 1 << 10
    cirrus_bit = 1 << 11
    qa_ok = (qa.bitwiseAnd(cloud_bit).eq(0)).And(qa.bitwiseAnd(cirrus_bit).eq(0))

    img = img.updateMask(scl_ok).updateMask(qa_ok)

    # scale reflectance to 0..1
    bands = ['B2','B3','B4','B5','B6','B7','B8','B8A','B11','B12']
    scaled = img.select(bands).multiply(0.0001)
    return img.addBands(scaled, overwrite=True).copyProperties(img, img.propertyNames())

# =====================================================
# FORCE ALL S2 BANDS TO 10 m GRID (resample 20m -> 10m)
# =====================================================
def s2_to_10m(img):
    p10 = img.select('B2').projection()

    b10 = img.select(['B2','B3','B4','B8'])
    b20 = (img.select(['B5','B6','B7','B8A','B11','B12'])
             .resample('bilinear')
             .reproject(p10.atScale(10)))

    return ee.Image.cat([b10, b20]).copyProperties(img, img.propertyNames())

# =====================================================
# EVI @ 10 m
# =====================================================
def add_evi(img):
    b2 = img.select('B2')
    b4 = img.select('B4')
    b8 = img.select('B8')
    evi = (b8.subtract(b4)
             .multiply(2.5)
             .divide(b8.add(b4.multiply(6.0)).subtract(b2.multiply(7.5)).add(1.0).add(eps))
             .rename('EVI'))
    return img.addBands(evi)

# =====================================================
# MEMORY-OPTIMIZED EVI TEXTURES @ 10 m (32 levels)
# =====================================================
def add_evi_textures(img):
    evi = img.select('EVI')

    # Quantize to 32 levels (0-31) to prevent Out of Memory (Error 8)
    evi_quant = (evi.clamp(-1, 1)
              .unitScale(-1, 1)
              .multiply(31)
              .toByte()
              .rename('EVI_q32'))

    evi_10 = evi_quant.reproject(img.select('B2').projection().atScale(EVI_TEX_SCALE))

    # Calculate textures on the memory-light matrix
    tex = evi_10.glcmTexture(size=EVI_TEX_SIZE)

    out = (tex.select(['EVI_q32_contrast', 'EVI_q32_dissimilarity', 'EVI_q32_entropy', 'EVI_q32_homogeneity'])
              .rename(['EVI_CON', 'EVI_DIS', 'EVI_ENT', 'EVI_HOM']))

    return img.addBands(out)

# =====================================================
# BUILD S2 COLLECTION
# =====================================================
s2 = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
      .filterBounds(pts_bbox)
      .filterDate(start_date, end_date)
      .map(mask_s2_sr_harmonized)
      .map(s2_to_10m)
      .map(add_evi))

# =====================================================
# 16-DAY BINNING (Median) + EVI Textures
# =====================================================
start = ee.Date(start_date)
end   = ee.Date(end_date)
n_bins = end.difference(start, 'day').divide(BIN_DAYS_EE).ceil()
bins = ee.List.sequence(0, n_bins.subtract(1))

S2_BANDS_OUT = [
    'B2','B3','B4','B5','B6','B7','B8','B8A','B11','B12',
    'EVI','EVI_CON','EVI_DIS','EVI_ENT','EVI_HOM'
]

def s2_bin_to_image(i):
    i = ee.Number(i)
    b_start = start.advance(i.multiply(BIN_DAYS_EE), 'day')
    b_end   = b_start.advance(BIN_DAYS_EE, 'day')

    ic_bin  = s2.filterDate(b_start, b_end)

    # NO CLIP HERE! Allows lazy evaluation at sample time
    comp = ic_bin.median()

    comp = add_evi_textures(comp)

    bin_mid = b_start.advance(HALF_BIN_DAYS_EE, 'day')

    return (comp
            .select(S2_BANDS_OUT)
            .set('n_imgs', ic_bin.size())
            .set('system:time_start', bin_mid.millis()))

s2_16d = (ee.ImageCollection(bins.map(s2_bin_to_image))
          .filter(ee.Filter.gt('n_imgs', 0))
          .sort('system:time_start'))

# =====================================================


In [ ]:
tasks_18 = export_bin_means(s2_16d, 'S2_2023', scale=10)

## Sentinel-2 RaoQ

The preprocessing logic below is adapted from the supplied notebook. Review its methods against the final manuscript before claiming exact numerical replication.

In [ ]:
import ee
# Initialize Earth Engine

# =====================================================
# REQUIRED INPUTS
# =====================================================

BUFFER_M   = 60
# Make sure table_field_data is defined in your notebook prior to running this
points_fc  = table_field_data

# Simple bbox around points (keeps memory low)
pts_bbox   = points_fc.geometry().buffer(BUFFER_M).bounds()

# =====================================================
# PARAMETERS
# =====================================================
S2_SCALE_SAMPLE = 10
S2_TILESCALE    = 16
RAOQ_SCALE      = 10
eps = 1e-6

BIN_DAYS_EE       = ee.Number(BIN_DAYS)
HALF_BIN_DAYS_EE  = BIN_DAYS_EE.divide(2)
NEG_ONE_DAY_EE    = ee.Number(-1)

# =====================================================
# S2 PREP & EVI FUNCTIONS
# =====================================================
def mask_s2_sr_harmonized(img):
    scl = img.select('SCL')
    scl_ok = (scl.neq(3).And(scl.neq(8)).And(scl.neq(9)).And(scl.neq(10)).And(scl.neq(11)))
    qa = img.select('QA60')
    qa_ok = (qa.bitwiseAnd(1 << 10).eq(0)).And(qa.bitwiseAnd(1 << 11).eq(0))
    img = img.updateMask(scl_ok).updateMask(qa_ok)

    bands = ['B2','B3','B4','B5','B6','B7','B8','B8A','B11','B12']
    scaled = img.select(bands).multiply(0.0001)
    return img.addBands(scaled, overwrite=True).copyProperties(img)

def s2_to_10m(img):
    p10 = img.select('B2').projection()
    b10 = img.select(['B2','B3','B4','B8'])
    b20 = img.select(['B5','B6','B7','B8A','B11','B12']).resample('bilinear').reproject(p10.atScale(10))
    return ee.Image.cat([b10, b20]).copyProperties(img)

def add_evi(img):
    b2, b4, b8 = img.select('B2'), img.select('B4'), img.select('B8')
    evi = (b8.subtract(b4)
             .multiply(2.5)
             .divide(b8.add(b4.multiply(6.0)).subtract(b2.multiply(7.5)).add(1.0).add(eps))
             .rename('EVI'))
    return img.addBands(evi)

# =====================================================
# THE CORRECTED RAO'S Q ALGORITHM (STRICT SYNTAX)
# =====================================================
def add_raoq_evi(img):
    img = img.clip(pts_bbox)

    evi = img.select('EVI').reproject(img.select('B2').projection().atScale(RAOQ_SCALE))

    # 3x3 neighborhood (radius 1)
    k = ee.Kernel.square(radius=1, units='pixels', normalize=False)
    arr = evi.neighborhoodToArray(k)

    # 1. Strict syntax: lengths must be an Image Array, and dimensions = 2
    lengths = ee.Image([9, 1]).toArray()
    X = arr.arrayReshape(lengths, 2)  # Shape: [9,1]
    Xt = X.arrayTranspose()           # Shape: [1,9]

    # 2. Stretch arrays to 9x9 before subtracting
    X_rep = X.arrayRepeat(1, 9)
    Xt_rep = Xt.arrayRepeat(0, 9)

    # 3. Compute pairwise absolute differences
    D = X_rep.subtract(Xt_rep).abs()

    # Mean across both axes leaves a 1x1 array
    Dmean = D.arrayReduce(ee.Reducer.mean(), [0, 1])

    # 4. Strict syntax: arrayGet strictly requires an Image of positions
    pos = ee.Image([0, 0])
    raoq = Dmean.arrayGet(pos).rename('RaoQ')

    return img.addBands(raoq)

# =====================================================
# BUILD & BIN COLLECTION (LIGHTWEIGHT)
# =====================================================
s2 = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
      .filterBounds(pts_bbox)
      .filterDate(start_date, end_date)
      .map(mask_s2_sr_harmonized)
      .map(s2_to_10m)
      .map(add_evi))

start = ee.Date(start_date)
end   = ee.Date(end_date)
n_bins = end.difference(start, 'day').divide(BIN_DAYS_EE).ceil()
bins = ee.List.sequence(0, n_bins.subtract(1))

# Only keeping EVI and RaoQ to save memory!
S2_BANDS_OUT = ['EVI', 'RaoQ']

def s2_bin_to_image(i):
    i = ee.Number(i)
    b_start = start.advance(i.multiply(BIN_DAYS_EE), 'day')
    b_end   = b_start.advance(BIN_DAYS_EE, 'day')

    ic_bin  = s2.filterDate(b_start, b_end)
    comp = ic_bin.median().clip(pts_bbox)

    # Calculate Rao's Q on the composite
    comp = add_raoq_evi(comp)

    bin_mid = b_start.advance(HALF_BIN_DAYS_EE, 'day')

    return (comp
            .select(S2_BANDS_OUT)
            .set('n_imgs', ic_bin.size())
            .set('system:time_start', bin_mid.millis()))

s2_16d = (ee.ImageCollection(bins.map(s2_bin_to_image))
          .filter(ee.Filter.gt('n_imgs', 0))
          .sort('system:time_start'))

# =====================================================


In [ ]:
tasks_20 = export_bin_means(s2_16d, 'RaoQ_2023', scale=10)

## 6. AlphaEarth annual embeddings

The embedding collection may have multiple spatial tiles. Mosaic all tiles for the study area, select the 64 documented AEF dimensions, and aggregate them to the same 17.84 m plot footprint. Verify year and band availability in your Earth Engine session.

In [ ]:
aef_collection = (ee.ImageCollection('GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL')
                  .filterDate(start_date,end_date).filterBounds(edaratkol_fc.geometry()))
if aef_collection.size().getInfo() == 0:
    raise ValueError('No AEF annual images found for this region and year.')
aef_bands = [f'A{i:02d}' for i in range(1,65)]
aef_image = aef_collection.mosaic().select(aef_bands).toFloat()
aef_rows = aef_image.reduceRegions(collection=plot_buffers,reducer=ee.Reducer.mean(),
                                   scale=10,tileScale=4)
aef_task = ee.batch.Export.table.toDrive(
    collection=aef_rows,description='AEF_2023_plot_means',folder=EXPORT_FOLDER,
    fileNamePrefix='AEF_2023_plot_means',fileFormat='CSV',selectors=['PLOTID']+aef_bands)
aef_task.start()
print('Started AEF export:',aef_task.id)

## 7. TESSERA input and local assembly

The manuscript obtained 2023 TESSERA tiles through a formal project request. This notebook cannot retrieve those tiles without that access. Process the authorized tiles to a private CSV containing `PLOTID` and **128** plot-mean embedding dimensions, using the same footprint and grid conventions. Keep the original tile provenance and version in your private methods log.

In [ ]:
# Run only after all EE tasks have completed and CSV files are downloaded.
# Set DIVERSITY_EXPORTS_DIR to the private directory of downloaded exports.
from glob import glob
EXPORTS_DIR = Path(os.environ.get('DIVERSITY_EXPORTS_DIR','private/exports'))
def load_bins(prefix):
    paths = sorted(EXPORTS_DIR.glob(f'{prefix}_bin_*.csv'))
    if len(paths) != 23:
        raise FileNotFoundError(f'Expected 23 {prefix} files; found {len(paths)} in {EXPORTS_DIR}')
    t = pd.concat((pd.read_csv(p,low_memory=False) for p in paths),ignore_index=True)
    if t.duplicated(['PLOTID','bin_start']).any():
        raise ValueError(f'{prefix}: duplicate plot–bin rows')
    return t.drop(columns=['.geo','system:index'],errors='ignore')

s1 = load_bins('S1_2023')
s2 = load_bins('S2_2023')
raoq = load_bins('RaoQ_2023')
if 'EVI' in raoq.columns:
    raoq = raoq.drop(columns=['EVI'])  # already exported with S2; avoid duplicate feature names
for name,t in [('S1',s1),('S2',s2),('RaoQ',raoq)]:
    print(name,len(t),'plots',t.PLOTID.nunique(),'bins',t.bin_start.nunique())
    t['PLOTID']=t.PLOTID.astype(str)
    t['bin_start']=t.bin_start.astype(str)
keys=['PLOTID','bin_start']
def predictors_only(table):
    return table.drop(columns=['bin','bin_end','n_imgs','time_start','bin_i'],errors='ignore')
long = predictors_only(s1).merge(predictors_only(s2),on=keys,how='outer',validate='one_to_one')
long = long.merge(predictors_only(raoq),on=keys,how='left',validate='one_to_one')
if long.duplicated(keys).any():raise ValueError('Repeated plot–bin keys after merge')
print('Joined long table:',long.shape,'plots:',long.PLOTID.nunique())

In [ ]:
aef_path=EXPORTS_DIR/'AEF_2023_plot_means.csv'
tessera_path=Path(os.environ.get('DIVERSITY_TESSERA_CSV','private/TESSERA_2023_plot_means.csv'))
for p in (aef_path,tessera_path):
    if not p.exists():raise FileNotFoundError(f'Provide authorized input: {p}')
aef=pd.read_csv(aef_path).drop(columns=['.geo','system:index'],errors='ignore')
tess=pd.read_csv(tessera_path).drop(columns=['.geo','system:index'],errors='ignore')
for name,t,n_expected in [('AEF',aef,64),('TESSERA',tess,128)]:
    if 'PLOTID' not in t:raise ValueError(f'{name} lacks PLOTID')
    t['PLOTID']=t.PLOTID.astype(str)
    if t.PLOTID.duplicated().any():raise ValueError(f'{name} has duplicate plot IDs')
    print(name,'available predictor dimensions:',len(t.columns)-1,'expected:',n_expected)
    if len(t.columns)-1 != n_expected:
        raise ValueError(f'{name}: review dimensions and exclude metadata columns before merging.')

# Targets and zone assignment come exclusively from the local restricted inventory.
inventory=plots.drop(columns=['Longitude','Latitude']).copy()
inventory.PLOTID=inventory.PLOTID.astype(str)
long.PLOTID=long.PLOTID.astype(str)
static=inventory.merge(aef,on='PLOTID',how='left',validate='one_to_one')
static=static.merge(tess,on='PLOTID',how='left',validate='one_to_one')
if static.isna().all(axis=0).any():
    raise ValueError('An input dimension contains only missing values.')
temporal=long.drop(columns=['bin_start','PLOTID']).columns.tolist()
wide=long.pivot(index='PLOTID',columns='bin_start',values=temporal)
wide.columns=[f'{name}_{date}' for name,date in wide.columns]
wide=wide.reset_index()
wide=static.merge(wide,on='PLOTID',how='left',validate='one_to_one')
out=Path(os.environ.get('DIVERSITY_WIDE_CSV','private/MODELING_DATASET_2023_WIDE.csv'))
out.parent.mkdir(parents=True,exist_ok=True)
wide.to_csv(out,index=False)
print('Saved private modeling table:',out,wide.shape)
print('Review the exported band names against the companion DNN notebook before running it.')